# Zhilicon Sovereign Inference — Hands-on Notebook

This notebook exercises the Zhilicon SDK from inside the attested pod that
`02-install-operator.sh` spun up. It shows:

1. Checking the attested sovereign context at process start.
2. Running a sovereign-aware inference-style workload.
3. Reading telemetry events (Tj, TRNG health, attestation rotations).

Run this from the pod shell:

```bash
kubectl -n bank-a exec -it <pod-name> -- python -m jupyter notebook --allow-root
```

Or open the notebook through the in-cluster Jupyter sidecar if your deployment enables one.

## 1. Read the attested sovereign context

The operator injects four environment variables into every managed pod. Read them first.

In [ ]:
import os

ctx = {
    'zone':         os.environ.get('ZHILICON_SOVEREIGN_ZONE'),
    'jurisdiction': os.environ.get('ZHILICON_JURISDICTION'),
    'endpoint':     os.environ.get('ZHILICON_ATTESTATION_ENDPOINT'),
    'proof_ref':    os.environ.get('ZHILICON_ATTESTATION_PROOF_REF'),
}
assert ctx['zone'] == 'uae-fintech', f"unexpected zone: {ctx['zone']}"
assert ctx['jurisdiction'] == 'ae',  f"unexpected jurisdiction: {ctx['jurisdiction']}"
print('Sovereign context:')
for k, v in ctx.items():
    print(f'  {k:13s} = {v}')

## 2. Run a sovereign-gated inference-style workload

We're in a pod whose scheduler only created it after the zone attestation landed.
From here, a real build would call `zhilicon.compute(...)` against the attested
silicon. For the demo we run a deterministic ECDSA batch that stands in for the
Sentinel-1 crypto engine.

In [ ]:
import time
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes

BATCH_SIZE = 4096
key = ec.generate_private_key(ec.SECP256R1())

t0 = time.monotonic()
for i in range(BATCH_SIZE):
    msg = f'bank-a/txn/{i:06d}'.encode()
    key.sign(msg, ec.ECDSA(hashes.SHA256()))
dt = time.monotonic() - t0
print(f'Signed {BATCH_SIZE} messages in {dt:.3f}s ({BATCH_SIZE/dt:.0f} sig/s)')

## 3. Read telemetry from the in-cluster Prometheus

The operator's metrics Service is at
`zhilicon-zhilicon-operator-metrics.zhilicon-system.svc.cluster.local:8080/metrics`.
If your cluster has Prometheus + Grafana installed, point Grafana at the
bundled `zhilicon-overview.json` dashboard and watch the live view. The
snippet below pulls the raw metrics for a sanity check.

In [ ]:
import urllib.request

URL = 'http://zhilicon-zhilicon-operator-metrics.zhilicon-system.svc.cluster.local:8080/metrics'
try:
    with urllib.request.urlopen(URL, timeout=5) as r:
        body = r.read().decode()
    interesting = [line for line in body.splitlines()
                   if line.startswith(('zhilicon_', 'controller_runtime_reconcile_'))]
    print('\n'.join(interesting[:30]))
except Exception as e:
    print(f'metrics endpoint not reachable from this pod: {e}')

## 4. Next steps

* **Plug into real silicon** — switch the device plugin to the `sysfs`
  discovery backend in `values.yaml` and redeploy. Re-run the notebook.
* **Scale out** — edit `manifests/devicepool.yaml:minDevices` upward and
  submit more workloads. The scheduler will pack them according to the
  pool's priority policy.
* **Add observability** — install kube-prometheus-stack, then enable
  `operator.metrics.serviceMonitor.enabled=true` on the Helm install so
  the operator publishes a ServiceMonitor automatically.
* **Harden attestation** — replace the demo signing key in the Kubernetes
  Secret with an HSM-backed key custodian (see
  `programs/portfolio-upgrade/RELIABILITY_SECURITY_SUPPLY_CHAIN.md` §2).
